# Version 2.3

- (2026.5.21) Fix a critical error: missing user_request in task
- (2026.5.21) Add a new tool get_region_allowed_settings as old check_regular_formula's docstrings is too long and redundant
- (2026.5.22) Abandon the transcript log as it may explode when agent is running
- (2026.5.22) CRITICAL ERROR FIX: add region response in `search_datafields`
- (2026.5.22) Give specific division for agent task: Deepseek/Kimi for main tool use / summary, Gemini for creative work
- (2026.5.22) Add creative agent (no tool use)
- (2026.5.22) Reduce redundant doc retriever output
- (2026.5.23) Fix info error in `get_region_allowed_settings`
- (2026.5.23) CRITICAL ERROR FIX: Max position error
- (2026.5.23) CRITICAL ERROR FIX: no function error handle in `_retry_after`
- (2026.5.23) Fix terminal log display problem (error in script, not in jupyter notebook)
- (2026.5.24) Fix the missing error message in `wqb_api`
- (2026.5.31) Add local, lightweight cross-encoder reranker
> You are initializing 5 separate vector databases, grabbing the top 8 matches from each ($k=8$), and passing all 40 context documents to the LLM via MergerRetriever. MergerRetriever simply alternates or chains documents chronologically from individual databases; it does not recalculate global relevance scores. Passing 40 raw, un-reranked snippets into the LLM context causes "lost-in-the-middle" token saturation. The researcher gets overwhelmed by redundant document noise, leading to degraded synthesis performance.
```python
from langchain_community.document_compressors import FlashrankRerank
from langchain_classic.retrievers import ContextualCompressionRetriever
# pip install flashrank
from flashrank import Ranker

# Build the base multi-database retriever
base_combined_retriever = Combine_Multiple_Embedding_Databases(EMBEDDING_DB_DIRECTORIES, embeddings)

# Initialize the Flashrank engine using your explicit 'Ranker' import
# This allows you to route the lightweight cross-encoder model through your existing HF cache directory
flashrank_client = Ranker(
    model_name="ms-marco-MiniLM-L-12-v2", 
    cache_dir=str(HF_CACHE_DIR)
)

# Wrap it in the LangChain Community Compressor component
compressor = FlashrankRerank(client=flashrank_client, top_n=5)

# 4. Finalize the pipeline using ContextualCompressionRetriever from langchain_classic
final_ranker_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, 
    base_retriever=base_combined_retriever
)
```

- (2026.5.31) Give the validator access to search_operators and search_datafields
> Your pipeline follows a strict sequential process: Researcher -> Ideator -> Coder -> Validator. If the validator runs an expression through wqb_simulate_api and receives an error stating a data field doesn't exist or syntax is fundamentally broken, it tries to self-correct up to 4 times. However, the validator does not have the tools to search the data fields or operators database (search_operators and search_datafields are only assigned to the coder). The validator is trapped in a loop where it must guess fixes blindly without access to information.

> Solution: The simplest and most effective solution is to give the validator access to search_operators and search_datafields. Since the validator is already executing the self-correction loop, giving it the tools directly eliminates unnecessary agent-to-agent overhead.

```python
validator = Agent(
    role="WorldQuant Submission Validator & Iterative Optimizer",
    goal="Ensure the alpha simulates correctly, passes all IS checks, and is ready to submit. Output ONLY in the exact user-specified format.",
    backstory="""You are the final gatekeeper and debugging expert. You never pass a broken alpha.
    
    Strictly follow the workflow:
    1. Use get_region_allowed_settings to confirm valid parameters.
    2. Use check_regular_formula to validate syntax and data fields.
    3. Use wqb_simulate_api and analyze the JSON output.
    4. If wqb_simulate_api or check_regular_formula returns an error (e.g., syntax broken, data field doesn't exist):
       - DO NOT GUESS. 
       - Immediately use `search_datafields` to find the correct field name or similar valid fields.
       - Immediately use `search_operators` to verify the exact syntax of the operator causing the failure.
    5. Iterate and fix the formula up to 4 times if needed.
    
    Never pass a broken alpha.""",
    # Add search_operators and search_datafields here:
    tools=[check_regular_formula, get_region_allowed_settings, wqb_simulate_api, search_operators, search_datafields],  
    llm=llm_pro,
    verbose=True,
    allow_delegation=False
)
```


- (TODO) CrewAI memory is weak by default. Add conversation memory.
```python
from crewai.memory import ShortTermMemory

# In Crew initialization
crew = Crew(
    ...
    memory=True,
    short_term_memory=ShortTermMemory(...)
)
```
- (OPTIONAL) ⭐ Transitioning from Sequential to Event-Driven (See in `materials/Transitioning from Sequential to Event-Driven.md`)
- (2026.5.27) "Output ONLY" Restraint Collides with Chain-of-Thought
> In task4, you instruct the validator: "STRICTLY OUTPUT RULE: Output ONLY the final working alpha. No extra explanations, no debugging notes...". Crushing the agent's output token generation boundaries prevents it from reasoning step-by-step through compilation errors. If an LLM cannot output its intermediate thoughts, its accuracy drops significantly when solving logical challenges.
- (TODO) Error output for html file in jupyter notebook

In [1]:
# Temp install command
# %pip install langchain langchain-text-splitters langchain-chroma langchain-openai langchain-community langchain-huggingface langchain-classic crewai crewai-tools sentence-transformers ipywidgets pypdf tqdm ansi2html flashrank